# 01 - Data Ingestion and Athena Catalog

**Owner:** Rajneesh |  **Project:** Customer Churn Prediction (Kaggle Customer Churn Dataset)

This notebook:
1. Uploads the two raw CSV files to the S3 data lake and records their checksums.
2. Creates a Glue database and Athena external tables so the data can be queried with SQL.
3. Validates row counts, null counts, and the number of labeled records per class against the course requirements.


In [2]:
%pip install -q awswrangler

Note: you may need to restart the kernel to use updated packages.


In [5]:
import sys, os, json, hashlib
sys.path.append("..")                      # make the src/ package importable

import boto3
import awswrangler as wr
from src import config

# BUCKET = config.get_bucket()
import boto3
from sagemaker.core.helper.session_helper import Session

session = boto3.session.Session()
region = session.region_name
sagemaker_session = Session()
BUCKET = sagemaker_session.default_bucket()

s3 = boto3.Session().client(service_name="s3", region_name=region)
ATHENA_OUT = f"s3://{BUCKET}/{config.ATHENA_RESULTS_PREFIX}/"
LOCAL_DIR = "../data/local"

print("Bucket:", BUCKET)
print("Athena output:", ATHENA_OUT)

Bucket: sagemaker-us-east-1-025824828736
Athena output: s3://sagemaker-us-east-1-025824828736/churn/athena-results/


## Step 1 - Upload raw files to S3 (raw data is kept immutable)

Each file goes in its own prefix (`train/`, `test/`) because an Athena table points at a folder.
An MD5 checksum is recorded so every experiment can be traced to the exact input files.

In [7]:
def md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

checksums = {}
for split, filename in config.FILES.items():
    local_path = os.path.join(LOCAL_DIR, filename)
    if not os.path.exists(local_path):
        raise FileNotFoundError(f"Download {filename} from Kaggle into {LOCAL_DIR}/ first")
    key = f"{config.PREFIXES[split]}/{filename}"
    s3.upload_file(local_path, BUCKET, key)
    checksums[filename] = md5(local_path)
    print(f"uploaded s3://{BUCKET}/{key}")

s3.put_object(Bucket=BUCKET, Key=f"{config.RAW_PREFIX}/checksums.json",
              Body=json.dumps(checksums, indent=2).encode())
checksums

uploaded s3://sagemaker-us-east-1-025824828736/churn/raw/train/customer_churn_dataset-training-master.csv
uploaded s3://sagemaker-us-east-1-025824828736/churn/raw/test/customer_churn_dataset-testing-master.csv


{'customer_churn_dataset-training-master.csv': '2932959fd769132d0e34e14d8e4ee2ef',
 'customer_churn_dataset-testing-master.csv': '55289b95d9b4310f554778c725097c5b'}

## Step 2 - Create the Glue database and Athena tables

Athena column names cannot contain spaces, so the table columns are snake_case
(for example `Usage Frequency` becomes `usage_frequency`). The CSV header row is skipped.

In [8]:
wr.catalog.create_database(config.GLUE_DATABASE, exist_ok=True)

DDL = '''
CREATE EXTERNAL TABLE IF NOT EXISTS {db}.{table} (
    customerid         double,
    age                double,
    gender             string,
    tenure             double,
    usage_frequency    double,
    support_calls      double,
    payment_delay      double,
    subscription_type  string,
    contract_length    string,
    total_spend        double,
    last_interaction   double,
    churn              double
)
ROW FORMAT DELIMITED FIELDS TERMINATED BY ','
STORED AS TEXTFILE
LOCATION 's3://{bucket}/{prefix}/'
TBLPROPERTIES ('skip.header.line.count'='1')
'''

for split, table in config.TABLES.items():
    sql = DDL.format(db=config.GLUE_DATABASE, table=table, bucket=BUCKET, prefix=config.PREFIXES[split])
    wr.athena.start_query_execution(sql=sql, database=config.GLUE_DATABASE, s3_output=ATHENA_OUT, wait=True)
    print("created table", table)

wr.catalog.tables(database=config.GLUE_DATABASE)

created table churn_train
created table churn_test


,Database,Table,Description,TableType,Columns,Partitions
0,customer_churn_db,churn_test,,EXTERNAL_TABLE,"customerid, age, gender, tenure, usage_frequen...",
1,customer_churn_db,churn_train,,EXTERNAL_TABLE,"customerid, age, gender, tenure, usage_frequen...",


## Step 3 - Validate the data with SQL

Checks against the course guidance: more than 10,000 rows per file and more than 10,000
labeled records per class.

In [9]:
def q(sql):
    return wr.athena.read_sql_query(sql, database=config.GLUE_DATABASE, s3_output=ATHENA_OUT, ctas_approach=False)

counts = {split: int(q(f"SELECT COUNT(*) AS n FROM {table}").n[0]) for split, table in config.TABLES.items()}
print("Rows per file:", counts)

for split, table in config.TABLES.items():
    print(f"\n--- {table}: churn class counts ---")
    print(q(f"SELECT churn, COUNT(*) AS n FROM {table} GROUP BY churn ORDER BY churn"))

[10/19/26 06:49:05] INFO     Initializing a Ray instance                                               ]8;id=16229272;file:///opt/conda/lib/python3.12/site-packages/awswrangler/distributed/ray/_core.py\_core.py]8;;\:]8;id=16229273;file:///opt/conda/lib/python3.12/site-packages/awswrangler/distributed/ray/_core.py#202\202]8;;\

2026-10-19 06:49:07,647	WARNING services.py:2213 -- WARNING: The object store is using /tmp/ray instead of /dev/shm because /dev/shm has only 906997760 bytes available. This will harm performance! You may be able to free up space by deleting files in /dev/shm. If you are inside a Docker container, you can increase /dev/shm size by passing '--shm-size=2.17gb' to 'docker run' (or add it to the run_options list in a Ray cluster config). Make sure to set this to more than 30% of available RAM.
2026-10-19 06:49:09,836	INFO worker.py:2012 -- Started a local Ray instance.
/opt/conda/lib/python3.12/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


Rows per file: {'train': 163284, 'test': 64374}

--- churn_train: churn class counts ---
   churn       n
0    0.0    2137
1    1.0  161146
2    NaN       1

--- churn_test: churn class counts ---
   churn      n
0    0.0  33881
1    1.0  30493


In [10]:
# Null counts per column (a null in a column shows up as a missing value in Athena)
cols = ["customerid","age","gender","tenure","usage_frequency","support_calls",
        "payment_delay","subscription_type","contract_length","total_spend","last_interaction","churn"]
null_sql = "SELECT " + ", ".join(f"SUM(CASE WHEN {c} IS NULL THEN 1 ELSE 0 END) AS {c}" for c in cols)
for split, table in config.TABLES.items():
    print(f"\n--- {table}: null counts ---")
    print(q(null_sql + f" FROM {table}").T.rename(columns={0: "nulls"}))


--- churn_train: null counts ---
                   nulls
customerid             0
age                    1
gender                 1
tenure                 1
usage_frequency        1
support_calls          1
payment_delay          1
subscription_type      1
contract_length        1
total_spend            1
last_interaction       1
churn                  1

--- churn_test: null counts ---
                   nulls
customerid             0
age                    0
gender                 0
tenure                 0
usage_frequency        0
support_calls          0
payment_delay          0
subscription_type      0
contract_length        0
total_spend            0
last_interaction       0
churn                  0


In [11]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>


I0000 00:00:1792392576.857966    2229 chttp2_transport.cc:1182] ipv4:169.255.255.2:40175: Got goaway [2] err=UNAVAILABLE:GOAWAY received; Error code: 2; Debug Text: Cancelling all calls {grpc_status:14, http2_error:2, created_time:"2026-10-19T06:49:36.857962242+00:00"}
*** SIGTERM received at time=1792392579 on cpu 1 ***
PC: @     0x7f28bb8ef2d2  (unknown)  epoll_wait
[2026-10-19 06:49:39,333 E 1403 1403] logging.cc:474: *** SIGTERM received at time=1792392579 on cpu 1 ***
[2026-10-19 06:49:39,333 E 1403 1403] logging.cc:474: PC: @     0x7f28bb8ef2d2  (unknown)  epoll_wait
